In [1]:
!pip install geobr
import pandas as pd
import numpy as np
import warnings
from google.colab import drive
import geobr
import geopandas
import matplotlib.pyplot as plt
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
dados = pd.read_excel('/content/drive/My Drive/covid/covid.xlsx')
dados.drop(['regiao', 'codRegiaoSaude','nomeRegiaoSaude', 'semanaEpi','Recuperadosnovos', 'emAcompanhamentoNovos', 'populacaoTCU2019', 'interior/metropolitana'],axis=1,inplace=True)
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 641652 entries, 0 to 641651
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   estado           641505 non-null  object        
 1   municipio        635100 non-null  object        
 2   coduf            641652 non-null  int64         
 3   codmun           637536 non-null  float64       
 4   data             641652 non-null  datetime64[ns]
 5   casosAcumulado   641652 non-null  int64         
 6   casosNovos       641652 non-null  int64         
 7   obitosAcumulado  641652 non-null  int64         
 8   obitosNovos      641652 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(5), object(2)
memory usage: 44.1+ MB


In [3]:
def clipar(x):
  if x == np.inf or x == -np.inf:
    return None
  return x

def clipar2(x):
  if x > 5:
    return None
  return x

janela = 7
janela2 = 14
janelaC = 5

dados['mediamovelcaso'] = 0
dados['mediamovelobito'] = 0
dados['crescimentoobito'] = 0.0
dados['crescimentocasos'] = 0

merge1 = dados[~dados.codmun.notnull() & ~dados.municipio.notnull()].copy()
merge1.sort_values(by=['data'],inplace=True)

estados = pd.DataFrame()
for c in merge1['coduf'].unique():

  estado = merge1[merge1.coduf==c].copy()
  estado['mediamovelcaso'] = estado.iloc[:,6].rolling(window=janela).mean()
  estado['mediamovelobito'] = estado.iloc[:,8].rolling(window=janela).mean()
  estado['crescimentocasos'] = np.clip(np.log10(estado['casosAcumulado']),0,6)

  estado['crescimentoobito'] = (estado.iloc[:,10]-estado.iloc[:,10].shift(janela2))/estado.iloc[:,10].shift(janela2)
  #estado['crescimentoobito'] = np.clip(estado['crescimentoobito'],-janelaC,janelaC)
  estado['crescimentoobito'].fillna(method='ffill', inplace=True)

  estados = pd.concat([estados, estado])

nacional = estados[estados.coduf==76]
estados = estados[estados.coduf!=76]
nacional.drop(['estado','municipio','coduf','codmun'],axis=1,inplace=True)
nacional.info()
nacional.to_csv('/content/drive/My Drive/covid/dados/nacional.csv', index = False, header=True)

estados.info()
estados.to_csv('/content/drive/My Drive/covid/dados/estadual.csv', index = False, header=True)

evolucao = estados.drop(['municipio', 'coduf', 'codmun', 'casosAcumulado', 'casosNovos', 'obitosAcumulado', 'obitosNovos', 'mediamovelcaso', 'mediamovelobito','crescimentocasos' ],axis=1)
evolucao = pd.pivot(evolucao,index='data',columns='estado',values='crescimentoobito')
for c in evolucao.columns:
  evolucao[c] = evolucao[c].apply(clipar)
evolucao = evolucao.transpose()
for c in evolucao.columns:
  evolucao[c] = evolucao[c].rank(method='min',ascending=False)
  evolucao[c] = evolucao[c].apply(clipar2)
evolucao = evolucao.transpose()
evolucao.info()
evolucao.tail(janela2).dropna(how='all', axis=1).to_csv('/content/drive/My Drive/covid/dados/evolucao.csv', header=True)

merge2 = dados[dados.codmun.notnull() & dados.municipio.notnull()].copy()
merge2.sort_values(by=['data'],inplace=True)

municipios = pd.DataFrame()
for c in merge2['codmun'].unique():

  municipio = merge2[merge2.codmun==c].copy()
  municipio['mediamovelcaso'] = municipio.iloc[:,6].rolling(window=janela).mean()
  municipio['mediamovelobito'] = municipio.iloc[:,8].rolling(window=janela).mean()
  municipio['crescimentocasos'] = np.clip(np.log10(municipio['casosAcumulado']),0,6)

  municipio['crescimentoobito'] = (municipio.iloc[:,10]-municipio.iloc[:,10].shift(janela2))/municipio.iloc[:,10].shift(janela2)
  municipio['crescimentoobito'] = np.clip(municipio['crescimentoobito'],-janelaC,janelaC)
  municipio['crescimentoobito'].fillna(method='ffill', inplace=True)

  municipios = pd.concat([municipios, municipio])

for e in estados['estado'].unique():
  municipios[municipios.estado==e].to_csv('/content/drive/My Drive/covid/dados/'+e+'.csv', index = False, header=True)
municipios.info()

#del dados
del municipio
del estado
del nacional
del evolucao
del merge2
del merge1

/usr/local/lib/python3.6/dist-packages/pandas/core/series.py:679: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


<class 'pandas.core.frame.DataFrame'>
Int64Index: 147 entries, 0 to 146
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   data              147 non-null    datetime64[ns]
 1   casosAcumulado    147 non-null    int64         
 2   casosNovos        147 non-null    int64         
 3   obitosAcumulado   147 non-null    int64         
 4   obitosNovos       147 non-null    int64         
 5   mediamovelcaso    141 non-null    float64       
 6   mediamovelobito   141 non-null    float64       
 7   crescimentoobito  126 non-null    float64       
 8   crescimentocasos  147 non-null    float64       
dtypes: datetime64[ns](1), float64(4), int64(4)
memory usage: 11.5 KB
<class 'pandas.core.frame.DataFrame'>
Int64Index: 3969 entries, 882 to 1175
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   estado    

In [4]:
import imageio
from os import listdir

def criagif(nome):
  filenames = ['/content/drive/My Drive/covid/img/'+nome+'/'+a for a in listdir('/content/drive/My Drive/covid/img/'+nome+'/')]
  with imageio.get_writer('/content/drive/My Drive/covid/'+nome+'.gif', mode='I') as writer:
      for filename in filenames:
          image = imageio.imread(filename)
          writer.append_data(image)

def cor(valor):
  if valor>=0.15:
    return 4
  elif valor<=-0.15:
    return 2
  elif abs(valor)<0.15:
    return 3
  else:
    return 1

def plotting(v,i,color,ax):
  if len(v[v['cor']==i])>0:
    v[v['cor']==i].plot(color=color,ax=ax)

def corta(i):
  return int(i) // 10

%matplotlib inline

def plota_nacional_mun(dias):

  for d in dias:

    mu = municipios[municipios.data==d]
    if len(mu)>0:
      fig, ax = plt.subplots(nrows=1, ncols=3,figsize=(9, 2.96), dpi=300)

      municipalities = geobr.read_municipality(code_muni='all')
      municipalities['code_muni']=municipalities['code_muni'].apply(corta)

      municipalities = municipalities.merge(mu,how='left',left_on='code_muni',right_on='codmun')
      municipalities['cor']=municipalities.crescimentoobito.apply(cor)

      municipalities.plot(column='crescimentocasos',cmap='Oranges', ax=ax[1],vmax=6)
      ax[1].axis('off')

      plotting(municipalities,1,'gray',ax[2])
      plotting(municipalities,2,'#99CCFF',ax[2])
      plotting(municipalities,3,'#FFFF66',ax[2])
      plotting(municipalities,4,'#FF9999',ax[2])
      ax[2].axis('off')
      fig.suptitle(pd.to_datetime(str(d)).strftime("%d/%m"), fontsize=10)

      locais=geobr.read_municipal_seat()
      estados=geobr.read_state(code_state='all')
      locais['code_muni']=locais['code_muni'].apply(corta)

      locais = locais.merge(mu,how='left',left_on='code_muni',right_on='codmun')
      locais['cor']=locais.crescimentoobito.apply(cor)
      estados.plot(ax=ax[0],edgecolor="black",facecolor="white",linewidth=0.5)

      locais.plot(column='crescimentocasos',cmap='Oranges', ax=ax[0], markersize='crescimentocasos', vmax=6)
      ax[0].axis('off')

      fig.savefig(f'/content/drive/My Drive/covid/img/nacional_mun/'+pd.to_datetime(str(d)).strftime("%m %d %Y")+'.png')
      plt.close()

def plota_nacional_state(dias):

  for d in dias:

    es = estados[estados.data==d]
    if len(es)>0:
      fig, ax = plt.subplots(nrows=1, ncols=2,figsize=(6, 2.96), dpi=300)

      states = geobr.read_state(code_state='all')
      states = states.merge(es,how='left',left_on='code_state',right_on='coduf')
      states['cor']=states.crescimentoobito.apply(cor)

      states.plot(column='crescimentocasos',cmap='Oranges', ax=ax[0],vmax=6)
      ax[0].axis('off')
      fig.suptitle(pd.to_datetime(str(d)).strftime("%d/%m"), fontsize=10)


      plotting(states,1,'gray',ax[1])
      plotting(states,2,'#99CCFF',ax[1])
      plotting(states,3,'#FFFF66',ax[1])
      plotting(states,4,'#FF9999',ax[1])
      ax[1].axis('off')

      fig.savefig(f'/content/drive/My Drive/covid/img/nacional/'+pd.to_datetime(str(d)).strftime("%m %d %Y")+'.png')
      plt.close()

def plota_estadual(dias):

  for d in dias:

    mu = municipios[municipios.data==d]
    if len(mu)>0:

      for es in estados['estado'].unique():

        fig, ax = plt.subplots(nrows=1, ncols=2,figsize=(6, 2.96), dpi=300)

        municipalities = geobr.read_municipality(code_muni=es)
        municipalities['code_muni']=municipalities['code_muni'].apply(corta)

        municipalities = municipalities.merge(mu,how='left',left_on='code_muni',right_on='codmun')
        municipalities['cor']=municipalities.crescimentoobito.apply(cor)

        municipalities.plot(column='crescimentocasos',cmap='Oranges', ax=ax[0],vmax=6)
        ax[0].axis('off')

        plotting(municipalities,1,'gray',ax[1])
        plotting(municipalities,2,'#99CCFF',ax[1])
        plotting(municipalities,3,'#FFFF66',ax[1])
        plotting(municipalities,4,'#FF9999',ax[1])
        ax[1].axis('off')
        fig.suptitle(pd.to_datetime(str(d)).strftime("%d/%m"), fontsize=10)

        fig.savefig(f'/content/drive/My Drive/covid/img/estadual/'+str(es)+'/'+pd.to_datetime(str(d)).strftime("%m %d %Y")+'.png')
        plt.close()
  

In [5]:
#plota_nacional_mun(estados['data'].unique())
#criagif('nacional_mun')
plota_nacional_state(estados['data'].unique())
criagif('nacional')
#plota_estadual(estados['data'].unique())
#for es in estados['estado'].unique():
#  criagif('estadual/'+str(es))

from __future__ import print_function  # for Python2
import sys

local_vars = list(locals().items())
for var, obj in local_vars:
    print(var, sys.getsizeof(obj))

__name__ 57
__doc__ 113
__package__ 16
__loader__ 16
__spec__ 16
__builtin__ 80
__builtins__ 80
_ih 128
_oh 240
_dh 72
_sh 80
In 128
Out 240
get_ipython 64
exit 56
quit 56
_ 53
__ 53
___ 53
_i 4246
_ii 3240
_iii 309
_i1 245
_exit_code 24
pd 80
np 80
warnings 80
drive 80
geobr 80
geopandas 80
plt 80
_i2 309
dados 144622131
_i3 3240
clipar 136
clipar2 136
janela 28
janela2 28
janelaC 28
estados 742227
c 32
municipios 148533964
e 51
_i4 4246
imageio 80
listdir 72
criagif 136
cor 136
plotting 136
corta 136
plota_nacional_mun 136
plota_nacional_state 136
plota_estadual 136
_i5 464
print_function 56
sys 80
